# 01 — Exploratory Data Analysis: LendingClub Credit Dataset

**GreenScore Climate-Adjusted CPD Engine**

This notebook performs exploratory data analysis on the LendingClub `accepted_2007_to_2018Q4.csv` dataset, covering:

1. **Data Shape & Types** — row/column counts, memory usage, dtypes
2. **Missing Values** — heatmap and per-column summary
3. **Target Distribution** — class balance of default vs. paid
4. **Univariate Distributions** — key numeric features
5. **Bivariate Analysis** — feature vs. default correlations
6. **Geographic Distribution** — loans by US state
7. **Correlation Matrix** — numeric feature inter-relationships

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = '../data/accepted_2007_to_2018Q4.csv'
print("Environment ready ✓")

## 1. Data Loading & Overview

In [ ]:
# Load with all columns for full exploration (sample 200k for speed)
df_raw = pd.read_csv(DATA_PATH, low_memory=False, nrows=200_000)

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Memory: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nColumn dtypes:")
print(df_raw.dtypes.value_counts())
df_raw.head(3)

In [ ]:
# Focus on the columns used by GreenScore
df = pd.read_csv(DATA_PATH, usecols=config.RAW_COLS, low_memory=False, nrows=200_000)

# Create binary target
df['default'] = df['loan_status'].apply(
    lambda x: 1 if str(x) in config.DEFAULT_STATUSES else 0
)
df['int_rate'] = df['int_rate'].astype(str).str.replace('%', '', regex=False).astype(float)
df['emp_length'] = df['emp_length'].astype(str).str.extract(r'(\d+)').astype(float).fillna(0)

print(f"Working dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.info()

## 2. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct}).sort_values('Percent', ascending=False)
missing_df = missing_df[missing_df['Count'] > 0]

if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=missing_df.index, y=missing_df['Percent'], ax=ax, color='coral')
    ax.set_title('Missing Values by Column (%)')
    ax.set_ylabel('Missing %')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    print(missing_df.to_string())
else:
    print("No missing values in selected columns ✓")

## 3. Target Distribution (Default vs. Paid)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = df['default'].value_counts()
labels = ['Paid (0)', 'Default (1)']
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Target Class Counts')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%', colors=colors,
            startangle=90, explode=(0, 0.05), shadow=True)
axes[1].set_title('Target Class Proportions')

plt.suptitle(f'Class Imbalance — Default Rate: {df["default"].mean():.1%}', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Univariate Feature Distributions

In [ ]:
numeric_cols = ['loan_amnt', 'dti', 'annual_inc', 'fico_range_low', 'int_rate', 'installment', 'emp_length']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = df[col].dropna()
    # Clip outliers at 99th percentile for visualization
    clip_val = data.quantile(0.99)
    data_clipped = data.clip(upper=clip_val)
    
    axes[i].hist(data_clipped, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(data.median(), color='red', linestyle='--', label=f'median={data.median():.0f}')
    axes[i].set_title(col)
    axes[i].legend(fontsize=9)

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Numeric Feature Distributions (clipped at 99th percentile)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics
df[numeric_cols].describe().round(2)

## 5. Bivariate Analysis — Feature vs. Default

In [ ]:
key_features = ['fico_range_low', 'dti', 'int_rate', 'annual_inc']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    data = df[[col, 'default']].dropna()
    data_clipped = data.copy()
    data_clipped[col] = data_clipped[col].clip(upper=data_clipped[col].quantile(0.99))
    
    sns.boxplot(data=data_clipped, x='default', y=col, ax=axes[i],
                palette=['#2ecc71', '#e74c3c'])
    axes[i].set_xticklabels(['Paid', 'Default'])
    axes[i].set_title(f'{col} by Default Status')

plt.suptitle('Feature Distributions: Paid vs. Default', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Default rate by categorical features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By home_ownership
ho_default = df.groupby('home_ownership')['default'].mean().sort_values(ascending=False)
ho_default.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Default Rate by Home Ownership')
axes[0].set_ylabel('Default Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# By purpose (top 10)
purpose_default = df.groupby('purpose')['default'].mean().sort_values(ascending=False).head(10)
purpose_default.plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Default Rate by Loan Purpose (Top 10)')
axes[1].set_xlabel('Default Rate')

# By FICO bucket
df['fico_group'] = pd.cut(df['fico_range_low'], bins=[300, 580, 670, 740, 850],
                           labels=['Poor\n300-580', 'Fair\n580-670', 'Good\n670-740', 'Excellent\n740-850'])
fico_default = df.groupby('fico_group', observed=True)['default'].mean()
fico_default.plot(kind='bar', ax=axes[2], color=['#e74c3c', '#f39c12', '#27ae60', '#2ecc71'], edgecolor='white')
axes[2].set_title('Default Rate by FICO Score Group')
axes[2].set_ylabel('Default Rate')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 6. Geographic Distribution of Loans

In [ ]:
# Loans by state and default rate by state
state_stats = df.groupby('addr_state').agg(
    loan_count=('default', 'count'),
    default_rate=('default', 'mean')
).sort_values('loan_count', ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Top 20 states by loan volume
top_states = state_stats.head(20)
axes[0].bar(top_states.index, top_states['loan_count'], color='steelblue', edgecolor='white')
axes[0].set_title('Top 20 States by Loan Volume')
axes[0].set_ylabel('Loan Count')

# Default rate by state (all)
state_default = state_stats.sort_values('default_rate', ascending=False)
colors = ['#e74c3c' if r > 0.20 else '#f39c12' if r > 0.15 else '#2ecc71' for r in state_default['default_rate']]
axes[1].barh(state_default.index, state_default['default_rate'], color=colors, edgecolor='white')
axes[1].axvline(df['default'].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Overall avg: {df["default"].mean():.1%}')
axes[1].set_title('Default Rate by State')
axes[1].set_xlabel('Default Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"States with highest default rates:")
print(state_default.head(10).to_string())

## 7. Correlation Matrix

In [ ]:
# Engineer features for correlation analysis
df['income_to_installment'] = df['annual_inc'] / (df['installment'] * 12 + 1)
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)

corr_cols = ['dti', 'annual_inc', 'fico_range_low', 'int_rate', 'installment',
             'emp_length', 'income_to_installment', 'loan_to_income', 'loan_amnt', 'default']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlations with default (sorted by absolute value)
default_corr = corr['default'].drop('default').abs().sort_values(ascending=False)
print("Feature correlations with default (|r|):")
for feat, val in default_corr.items():
    direction = "+" if corr.loc[feat, 'default'] > 0 else "-"
    print(f"  {feat:25s} {direction}{val:.4f}")

## 8. Key Findings Summary

| Observation | Detail |
|---|---|
| **Class Imbalance** | ~18% default rate — handled via `scale_pos_weight` in XGBoost |
| **FICO Score** | Strongest signal — default rate drops dramatically with higher scores |
| **Interest Rate** | Positively correlated with default — riskier borrowers pay more |
| **DTI** | Moderate positive correlation with default |
| **Annual Income** | Weak negative correlation — higher income slightly reduces risk |
| **Geographic Spread** | CA, TX, NY dominate volume; some smaller states show elevated default rates |
| **Loan Purpose** | Small business and renewable energy loans carry higher default risk |

These patterns justify the feature engineering in `config.py` (FICO buckets, DTI buckets, income-to-installment ratio) and inform the physical/transition risk overlay design.